# 02 · Data cleaning
Input: the untouched files in `raw/` (from notebook 01). Output:
- `interim/`: parsed but uncleaned tables (one per source), so every later step can be compared with the starting point
- `clean/delhi_pm25_hourly.parquet`: the final hourly table used for analysis
- `interim/station_match_cpcb_openaq.csv`: which OpenAQ location is the same station as each CPCB station
- `metadata/cleaning_log.csv`: one line per cleaning step: what we did, how many rows it touched, and why
- `metadata/data_dictionary.csv`: every column of the final table: meaning, type, unit, source, allowed and observed range, % missing

The study window is **15 September – 30 November** of 2023, 2024 and 2025. All times are **Indian Standard Time (IST)**, and each timestamp marks the **start** of the hour.

In [1]:
import json, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW, INTERIM, CLEAN, META = ROOT / "raw", ROOT / "interim", ROOT / "clean", ROOT / "metadata"
INTERIM.mkdir(exist_ok=True); CLEAN.mkdir(exist_ok=True)

YEARS = [2023, 2024, 2025]
DIWALI = {2023: "2023-11-12", 2024: "2024-10-31", 2025: "2025-10-20"}

def in_window(ts):
    """True for timestamps between 15 Sep and 30 Nov (inclusive) of the same year."""
    md = ts.dt.month * 100 + ts.dt.day
    return (md >= 915) & (md <= 1130)

LOG = []
def log(step, action, rows, reason):
    """Record one cleaning step. rows = number of rows (or values) affected."""
    LOG.append({"step": len(LOG) + 1, "stage": step, "what_we_did": action, "rows_affected": int(rows), "why": reason})
    print(f"[{len(LOG):02d}] {step}: {action} -> {int(rows):,} rows | {reason}")

## 1 · Load and parse each source into a tidy table (→ `interim/`)
### 1a · CPCB (36 yearly files → one long table)
The CPCB column names carry their unit in brackets (`PM2.5 (µg/m³)`, `WS (m/s)`). We rename them to short names and keep the units in the data dictionary instead. We keep only the columns we need: PM2.5, PM10 and the station's own weather sensors.

In [2]:
CPCB_NAMES = {"anand_vihar": "Anand Vihar, Delhi - DPCC", "rk_puram": "R K Puram, Delhi - DPCC", "okhla_ph2": "Okhla Phase-2, Delhi - DPCC",
              "punjabi_bagh": "Punjabi Bagh, Delhi - DPCC", "najafgarh": "Najafgarh, Delhi - DPCC", "ito": "ITO, Delhi - CPCB",
              "pusa_dpcc": "Pusa, Delhi - DPCC", "pusa_iitm": "Pusa, Delhi - IITM", "lodhi_road_imd": "IMD Lodhi Road, Delhi - IITM",
              "jahangirpuri": "Jahangirpuri, Delhi - DPCC", "narela": "Narela, Delhi - DPCC", "dwarka_sec8": "Dwarka-Sector 8, Delhi - DPCC"}
RENAME = {"Timestamp": "ts_ist", "PM2.5 (µg/m³)": "pm25", "PM10 (µg/m³)": "pm10",
          "AT (°C)": "st_temp_c", "RH (%)": "st_rh_pct", "WS (m/s)": "st_wind_ms", "WD (deg)": "st_wind_dir_deg"}

frames = []
for f in sorted((RAW / "cpcb").glob("cpcb_site*.csv")):
    site_id, key, year = re.match(r"cpcb_site(\d+)_(.+)_(\d{4})\.csv", f.name).groups()
    d = pd.read_csv(f)
    raw_cols = len(d.columns)
    d = d.rename(columns=RENAME)[list(RENAME.values())]
    d.insert(0, "station_key", key); d.insert(1, "cpcb_site_id", int(site_id))
    frames.append(d)
cpcb_raw = pd.concat(frames, ignore_index=True)
print("All CPCB rows (full calendar years):", cpcb_raw.shape)
log("Formats", "CPCB: renamed the 7 columns we keep from 'NAME (unit)' to short names (e.g. 'PM2.5 (µg/m³)' -> pm25); dropped the other 18 columns (gases, VOCs, rain, radiation, pressure)",
    len(cpcb_raw), "Units go in the data dictionary, not in column names; the dropped variables are not part of the question")

All CPCB rows (full calendar years): (315648, 9)
[01] Formats: CPCB: renamed the 7 columns we keep from 'NAME (unit)' to short names (e.g. 'PM2.5 (µg/m³)' -> pm25); dropped the other 18 columns (gases, VOCs, rain, radiation, pressure) -> 315,648 rows | Units go in the data dictionary, not in column names; the dropped variables are not part of the question


In [3]:
# Timestamps: text 'YYYY-MM-DD HH:MM:SS' in IST, with no time-zone marker. Check every value parses with the same format.
before_type = cpcb_raw["ts_ist"].dtype
cpcb_raw["ts_ist"] = pd.to_datetime(cpcb_raw["ts_ist"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
bad_ts = cpcb_raw["ts_ist"].isna().sum()
print(f"Timestamp dtype {before_type} -> {cpcb_raw['ts_ist'].dtype}; values that failed to parse: {bad_ts}")
log("Formats", f"CPCB: converted the Timestamp text to dates (IST, start of hour); {bad_ts} values failed", len(cpcb_raw), "Text dates cannot be sorted or joined")

# Number strings: check that no value column was read as text
non_numeric = {c: int(pd.to_numeric(cpcb_raw[c], errors="coerce").isna().sum() - cpcb_raw[c].isna().sum())
               for c in ["pm25", "pm10", "st_temp_c", "st_rh_pct", "st_wind_ms", "st_wind_dir_deg"]}
print("Non-numeric strings found per column:", non_numeric)

# Keep only the study window
n0 = len(cpcb_raw)
cpcb = cpcb_raw[in_window(cpcb_raw["ts_ist"])].copy()
log("Scope", f"CPCB: removed rows outside 15 Sep - 30 Nov ({n0:,} -> {len(cpcb):,} rows)", n0 - len(cpcb), "Outside the study window")
cpcb.to_parquet(INTERIM / "cpcb_hourly_parsed.parquet", index=False)
cpcb.head()

Timestamp dtype str -> datetime64[us]; values that failed to parse: 0
[02] Formats: CPCB: converted the Timestamp text to dates (IST, start of hour); 0 values failed -> 315,648 rows | Text dates cannot be sorted or joined
Non-numeric strings found per column: {'pm25': 0, 'pm10': 0, 'st_temp_c': 0, 'st_rh_pct': 0, 'st_wind_ms': 0, 'st_wind_dir_deg': 0}
[03] Scope: CPCB: removed rows outside 15 Sep - 30 Nov (315,648 -> 66,528 rows) -> 249,120 rows | Outside the study window


,station_key,cpcb_site_id,ts_ist,pm25,pm10,st_temp_c,st_rh_pct,st_wind_ms,st_wind_dir_deg
6168,pusa_iitm,107,2023-09-15 00:00:00,57.0775,68.1075,NaN,NaN,NaN,NaN
6169,pusa_iitm,107,2023-09-15 01:00:00,38.5850,58.5475,NaN,NaN,NaN,NaN
6170,pusa_iitm,107,2023-09-15 02:00:00,43.8950,73.2150,NaN,NaN,NaN,NaN
6171,pusa_iitm,107,2023-09-15 03:00:00,38.9975,54.3275,NaN,NaN,NaN,NaN
6172,pusa_iitm,107,2023-09-15 04:00:00,31.6775,54.3125,NaN,NaN,NaN,NaN


### 1b · OpenAQ (1,008 daily files → one long table of 15-minute PM2.5 readings)

In [4]:
oa = pd.concat([pd.read_csv(f) for f in sorted((RAW / "openaq").glob("loc*/*.csv.gz"))], ignore_index=True)
print("All OpenAQ readings, all parameters:", oa.shape)
n0 = len(oa)
oa = oa[oa["parameter"] == "pm25"].copy()
log("Scope", f"OpenAQ: removed readings of other parameters, kept PM2.5 only ({n0:,} -> {len(oa):,} readings)", n0 - len(oa), "Other pollutants and weather are outside the question")

print("Location names as OpenAQ writes them (note the number tacked on the end):")
print(oa.groupby("location_id")["location"].first().to_string())

All OpenAQ readings, all parameters: (803945, 9)
[04] Scope: OpenAQ: removed readings of other parameters, kept PM2.5 only (803,945 -> 80,525 readings) -> 723,420 rows | Other pollutants and weather are outside the question
Location names as OpenAQ writes them (note the number tacked on the end):
location_id
17              R K Puram, Delhi - DPCC-3379481
50           Punjabi Bagh, Delhi - DPCC-3379482
235       Anand Vihar, New Delhi - DPCC-3379575
5404                  Pusa, Delhi - IMD-3379470
5613              ITO, New Delhi - CPCB-3379477
5634        Lodhi Road, New Delhi - IMD-3379472
6356                 Pusa, Delhi - DPCC-3379541
6931     Dwarka-Sector 8, Delhi - DPCC -3379507
8235         Jahangirpuri, Delhi - DPCC-3379508
8239        Okhla Phase-2, Delhi - DPCC-3379513
10485              Narela, Delhi - DPCC-3379511
10488           Najafgarh, Delhi - DPCC-3379512
11607          Lodhi Road, Delhi - IITM-3379667


In [5]:
# Time zone: OpenAQ times are ISO strings with an explicit +05:30 offset. Convert to timezone-aware, then drop the
# offset so they line up with CPCB's plain IST times.
oa["ts_ist"] = pd.to_datetime(oa["datetime"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None)
offsets = oa["datetime"].str[-6:].value_counts()
print("UTC offsets found in the raw strings:", offsets.to_dict())
log("Formats", "OpenAQ: converted timestamps with a +05:30 offset to plain IST, the same clock as CPCB", len(oa), "The two sources must use the same clock before they can be compared")

# Station names: strip the numeric suffix and the 'New ' spelling difference
oa["name_raw"] = oa["location"]
oa["name_clean"] = oa["location"].str.replace(r"-\d+$", "", regex=True).str.replace("New Delhi", "Delhi").str.strip()
n_changed = (oa["name_raw"] != oa["name_clean"]).sum()
log("Formats", "OpenAQ: cleaned station names, removing the numeric suffix ('-3379575') and changing 'New Delhi' to 'Delhi'", n_changed, "So names can be compared with CPCB's spelling")
print(oa.groupby("location_id")[["name_raw", "name_clean"]].first().to_string())

UTC offsets found in the raw strings: {'+05:30': 80525}
[05] Formats: OpenAQ: converted timestamps with a +05:30 offset to plain IST, the same clock as CPCB -> 80,525 rows | The two sources must use the same clock before they can be compared
[06] Formats: OpenAQ: cleaned station names, removing the numeric suffix ('-3379575') and changing 'New Delhi' to 'Delhi' -> 80,525 rows | So names can be compared with CPCB's spelling
                                           name_raw                     name_clean
location_id                                                                       
17                  R K Puram, Delhi - DPCC-3379481        R K Puram, Delhi - DPCC
50               Punjabi Bagh, Delhi - DPCC-3379482     Punjabi Bagh, Delhi - DPCC
235           Anand Vihar, New Delhi - DPCC-3379575      Anand Vihar, Delhi - DPCC
5404                      Pusa, Delhi - IMD-3379470              Pusa, Delhi - IMD
5613                  ITO, New Delhi - CPCB-3379477              ITO, Delhi

In [6]:
# Exact duplicates in OpenAQ (same sensor, same time, same value)
dups = oa.duplicated(["location_id", "sensors_id", "datetime"]).sum()
oa = oa.drop_duplicates(["location_id", "sensors_id", "datetime"])
log("Duplicates", "OpenAQ: checked for repeated readings (same sensor and time); none found", dups, "A reading should appear once")

# Daily files overlap because they are cut in UTC; keep the window only
n0 = len(oa)
oa = oa[in_window(oa["ts_ist"])]
log("Scope", "OpenAQ: removed readings outside 15 Sep - 30 Nov 2025 (the daily files overlap by a few hours)", n0 - len(oa), "Outside the study window")

# 15-minute readings -> hourly means (an hour = readings stamped hh:00, hh:15, hh:30, hh:45)
oa["hour"] = oa["ts_ist"].dt.floor("h")
oa_hourly = (oa.groupby(["location_id", "name_clean", "hour"])["value"]
               .agg(pm25="mean", n_readings="size").reset_index().rename(columns={"hour": "ts_ist"}))
log("Formats", f"OpenAQ: averaged 15-minute readings into hourly means ({len(oa):,} readings -> {len(oa_hourly):,} hours)", len(oa) - len(oa_hourly),
    "CPCB reports hourly averages; both sources need the same time step")
oa_hourly.to_parquet(INTERIM / "openaq_pm25_hourly_parsed.parquet", index=False)
oa_hourly["n_readings"].value_counts().sort_index()

[07] Duplicates: OpenAQ: checked for repeated readings (same sensor and time); none found -> 0 rows | A reading should appear once
[08] Scope: OpenAQ: removed readings outside 15 Sep - 30 Nov 2025 (the daily files overlap by a few hours) -> 1,981 rows | Outside the study window
[09] Formats: OpenAQ: averaged 15-minute readings into hourly means (78,544 readings -> 21,905 hours) -> 56,639 rows | CPCB reports hourly averages; both sources need the same time step


n_readings
1     1323
2      978
3     3157
4    16441
5        6
Name: count, dtype: int64

### 1c · Open-Meteo (3 JSON files → one hourly weather table)

In [7]:
wx = []
for f in sorted((RAW / "openmeteo").glob("*.json")):
    j = json.loads(f.read_text())
    wx.append(pd.DataFrame(j["hourly"]))
wx = pd.concat(wx, ignore_index=True)
wx["ts_ist"] = pd.to_datetime(wx.pop("time"), format="%Y-%m-%dT%H:%M")
wx = wx.rename(columns={"temperature_2m": "temp_c", "relative_humidity_2m": "rh_pct", "precipitation": "rain_mm",
                        "wind_speed_10m": "wind_kmh", "wind_direction_10m": "wind_dir_deg",
                        "surface_pressure": "pressure_hpa", "boundary_layer_height": "blh_m"})
print("Weather table:", wx.shape, "| missing values per column:", wx.isna().sum().to_dict())
wx.to_parquet(INTERIM / "openmeteo_hourly_parsed.parquet", index=False)
wx.describe().round(1)

Weather table: (5544, 8) | missing values per column: {'temp_c': 0, 'rh_pct': 0, 'rain_mm': 0, 'wind_kmh': 0, 'wind_dir_deg': 0, 'pressure_hpa': 0, 'blh_m': 0, 'ts_ist': 0}


,temp_c,rh_pct,rain_mm,wind_kmh,wind_dir_deg,pressure_hpa,blh_m,ts_ist
count,5544.0,5544.0,5544.0,5544.0,5544.0,5544.0,5544.0,5544
mean,24.6,63.0,0.0,7.1,227.9,986.7,434.8,2024-10-23 03:30:00
min,11.2,22.0,0.0,0.0,1.0,975.9,10.0,2023-09-15 00:00:00
25%,20.8,48.0,0.0,4.8,138.0,983.2,50.0,2023-11-11 17:45:00
50%,24.8,64.0,0.0,6.6,273.0,987.3,115.0,2024-10-23 11:30:00
75%,28.6,78.0,0.0,8.7,302.0,989.9,710.0,2025-10-04 05:15:00
max,38.7,99.0,16.1,27.6,360.0,996.9,2790.0,2025-11-30 23:00:00
std,5.4,18.6,0.4,3.5,98.6,4.2,572.8,NaN


## 2 · Units: one unit per quantity
CPCB measures wind in **m/s**, Open-Meteo in **km/h**. We convert CPCB wind to km/h (× 3.6) so the two can be compared directly.

In [8]:
n = cpcb["st_wind_ms"].notna().sum()
cpcb["st_wind_kmh"] = cpcb.pop("st_wind_ms") * 3.6
log("Units", "CPCB: converted station wind speed from m/s to km/h (x 3.6)", n, "Open-Meteo wind is in km/h; one unit per quantity")

[10] Units: CPCB: converted station wind speed from m/s to km/h (x 3.6) -> 48,865 rows | Open-Meteo wind is in km/h; one unit per quantity


## 3 · Duplicates
### 3a · Exact duplicates inside CPCB

In [9]:
d_cp = cpcb.duplicated(["station_key", "ts_ist"]).sum()
cpcb = cpcb.drop_duplicates(["station_key", "ts_ist"])
log("Duplicates", "CPCB: checked for repeated station-hours; none found", d_cp, "Each station-hour should appear once")

# Check every station has a complete hourly calendar (no missing timestamps, only missing values)
expected = len(pd.date_range("2023-09-15", "2023-11-30 23:00", freq="h"))
print("Hours per station-year:", cpcb.groupby(["station_key", cpcb.ts_ist.dt.year]).size().unique(), "| expected:", expected)

[11] Duplicates: CPCB: checked for repeated station-hours; none found -> 0 rows | Each station-hour should appear once
Hours per station-year: [1848] | expected: 1848


### 3b · Duplicates across sources: the same station in OpenAQ and CPCB (2025)
The two sources spell station names differently (e.g. OpenAQ "Pusa, New Delhi - IMD" vs CPCB "Pusa, Delhi - IITM"), so names alone are not a safe key.

**Matching rule:** an OpenAQ location and a CPCB station are the **same station** if, over the 2025 hours both have, their hourly PM2.5 values agree within 0.5 µg/m³ in at least 80% of hours. If no OpenAQ location reaches 80%, the one with the smallest mean absolute difference is accepted only if that difference is under 10 µg/m³ and the correlation is above 0.9.

In [10]:
cp25 = cpcb[cpcb.ts_ist.dt.year == 2025].pivot(index="ts_ist", columns="station_key", values="pm25")
oa_w = oa_hourly.pivot(index="ts_ist", columns="location_id", values="pm25")
pairs = []
for k in cp25.columns:
    for L in oa_w.columns:
        j = pd.concat([cp25[k], oa_w[L]], axis=1, keys=["cp", "oa"]).dropna()
        if len(j) < 100:
            continue
        diff = (j.cp - j.oa).abs()
        pairs.append({"station_key": k, "openaq_id": L, "hours_compared": len(j), "share_within_0.5": diff.lt(0.5).mean(),
                      "mean_abs_diff": diff.mean(), "corr": j.corr().iloc[0, 1]})
pairs = pd.DataFrame(pairs)
best = pairs.sort_values("mean_abs_diff").groupby("station_key").head(1).set_index("station_key")
best["rule"] = np.where(best["share_within_0.5"] >= 0.8, "exact (>=80% within 0.5)",
                np.where((best["mean_abs_diff"] < 10) & (best["corr"] > 0.9), "close (MAD<10, r>0.9)", "NO MATCH"))
names = oa_hourly.groupby("location_id")["name_clean"].first()
best["openaq_name"] = best["openaq_id"].map(names)
best["cpcb_name"] = best.index.map(CPCB_NAMES)
match = best[["cpcb_name", "openaq_id", "openaq_name", "hours_compared", "share_within_0.5", "mean_abs_diff", "corr", "rule"]].round(3)
match.to_csv(INTERIM / "station_match_cpcb_openaq.csv")
match

,cpcb_name,openaq_id,openaq_name,hours_compared,share_within_0.5,mean_abs_diff,corr,rule
station_key,,,,,,,,
okhla_ph2,"Okhla Phase-2, Delhi - DPCC",8239,"Okhla Phase-2, Delhi - DPCC",1698,1.000,0.000,1.000,exact (>=80% within 0.5)
jahangirpuri,"Jahangirpuri, Delhi - DPCC",8235,"Jahangirpuri, Delhi - DPCC",1643,1.000,0.000,1.000,exact (>=80% within 0.5)
dwarka_sec8,"Dwarka-Sector 8, Delhi - DPCC",6931,"Dwarka-Sector 8, Delhi - DPCC",1739,1.000,0.000,1.000,exact (>=80% within 0.5)
najafgarh,"Najafgarh, Delhi - DPCC",10488,"Najafgarh, Delhi - DPCC",1671,1.000,0.000,1.000,exact (>=80% within 0.5)
anand_vihar,"Anand Vihar, Delhi - DPCC",235,"Anand Vihar, Delhi - DPCC",1701,0.999,0.004,1.000,exact (>=80% within 0.5)
rk_puram,"R K Puram, Delhi - DPCC",17,"R K Puram, Delhi - DPCC",1730,0.989,0.031,1.000,exact (>=80% within 0.5)
punjabi_bagh,"Punjabi Bagh, Delhi - DPCC",50,"Punjabi Bagh, Delhi - DPCC",1628,0.999,0.142,0.999,exact (>=80% within 0.5)
pusa_dpcc,"Pusa, Delhi - DPCC",6356,"Pusa, Delhi - DPCC",1746,0.999,0.286,0.996,exact (>=80% within 0.5)
narela,"Narela, Delhi - DPCC",10485,"Narela, Delhi - DPCC",1758,0.909,0.595,1.000,exact (>=80% within 0.5)


Every CPCB station has an OpenAQ twin. For most, the values are **identical** hour by hour: the CPCB hourly figure is simply the mean of OpenAQ's four 15-minute readings. OpenAQ is a copy of CPCB data, not an independent measurement.

Where the twins differ slightly (IMD-run stations such as Pusa IITM and Lodhi Road), it is because CPCB's hourly average and the OpenAQ readings were processed separately; the correlation is still very high.

**Decision:** for 2025, keep the CPCB value and drop the OpenAQ copy. OpenAQ is used only to fill hours where CPCB is empty but OpenAQ has a reading. The OpenAQ location 11607 ("Lodhi Road, Delhi - IITM") matched none of our CPCB stations (it is a different monitor ~0.6 km away) and is dropped.

In [11]:
m = match[match["rule"] != "NO MATCH"].reset_index()[["station_key", "openaq_id"]]
oa_m = oa_hourly.merge(m, left_on="location_id", right_on="openaq_id")[["station_key", "ts_ist", "pm25"]]
both = cpcb.merge(oa_m, on=["station_key", "ts_ist"], how="inner", suffixes=("", "_oa"))
n_dup = both["pm25"].notna().sum() - 0
fill = both["pm25"].isna() & both["pm25_oa"].notna()
log("Duplicates", "Across sources: matched each CPCB station to its OpenAQ twin by value (rule: agree within 0.5 µg/m³ in >= 80% of hours); dropped the OpenAQ copy of every station-hour CPCB also has",
    (both["pm25"].notna() & both["pm25_oa"].notna()).sum(), "Same station, same hour, same measurement in two sources")
log("Duplicates", "Across sources: dropped OpenAQ location 11607 (Lodhi Road, IITM), which matched none of our stations", (oa_hourly["location_id"] == 11607).sum(), "A different monitor, not one of our 12")

cpcb = cpcb.merge(oa_m.rename(columns={"pm25": "pm25_oa"}), on=["station_key", "ts_ist"], how="left")
cpcb["pm25_source"] = np.where(cpcb["pm25"].notna(), "cpcb", None)
filled = cpcb["pm25"].isna() & cpcb["pm25_oa"].notna()
cpcb.loc[filled, "pm25"] = cpcb.loc[filled, "pm25_oa"]
cpcb.loc[filled, "pm25_source"] = "openaq"
log("Missing values", "Filled empty CPCB hours in 2025 with the matched OpenAQ hourly value", filled.sum(), "Same station's data published through a second channel")
cpcb = cpcb.drop(columns="pm25_oa")

[12] Duplicates: Across sources: matched each CPCB station to its OpenAQ twin by value (rule: agree within 0.5 µg/m³ in >= 80% of hours); dropped the OpenAQ copy of every station-hour CPCB also has -> 20,120 rows | Same station, same hour, same measurement in two sources
[13] Duplicates: Across sources: dropped OpenAQ location 11607 (Lodhi Road, IITM), which matched none of our stations -> 1,585 rows | A different monitor, not one of our 12
[14] Missing values: Filled empty CPCB hours in 2025 with the matched OpenAQ hourly value -> 200 rows | Same station's data published through a second channel


### 3c · Same site, two names: Pusa (DPCC) and Pusa (IITM)
OpenAQ places "Pusa, Delhi - DPCC" and "Pusa, Delhi - IMD" at the **same coordinates** (28.639645, 77.14626). In CPCB they are two separate monitors. Their readings are not copies of each other, but they measure the same air. If we kept both, the Pusa site would count twice in any city-wide average.

**Rule:** merge them into one site, `pusa`, by averaging the two hourly values; if only one has a value, use it.

In [12]:
pusa = cpcb[cpcb.station_key.isin(["pusa_dpcc", "pusa_iitm"])].pivot(index="ts_ist", columns="station_key", values="pm25")
print("Pusa DPCC vs IITM: correlation by year")
print(pusa.groupby(pusa.index.year).corr().xs("pusa_dpcc", level=1)["pusa_iitm"].round(3).to_string())
print("Exactly equal hours:", (pusa.pusa_dpcc.round(1) == pusa.pusa_iitm.round(1)).sum(), "of", len(pusa.dropna()))

agg = {c: "mean" for c in ["pm25", "pm10", "st_temp_c", "st_rh_pct", "st_wind_kmh", "st_wind_dir_deg"]}
pm = cpcb[cpcb.station_key.str.startswith("pusa")].groupby("ts_ist").agg(
    {**agg, "pm25_source": lambda s: "cpcb" if (s == "cpcb").any() else ("openaq" if (s == "openaq").any() else None)}).reset_index()
pm["station_key"] = "pusa"; pm["cpcb_site_id"] = -1
n_before = len(cpcb)
cpcb = pd.concat([cpcb[~cpcb.station_key.str.startswith("pusa")], pm], ignore_index=True)
log("Duplicates", "Merged 'Pusa, Delhi - DPCC' and 'Pusa, Delhi - IITM' into one site 'pusa' (hourly mean of the two); 12 stations -> 11 sites",
    n_before - len(cpcb), "Same location under two names; keeping both would count the site twice")
print("Stations now:", sorted(cpcb.station_key.unique()))

Pusa DPCC vs IITM: correlation by year
ts_ist
2023    0.852
2024    0.937
2025    0.866
Exactly equal hours: 8 of 5354
[15] Duplicates: Merged 'Pusa, Delhi - DPCC' and 'Pusa, Delhi - IITM' into one site 'pusa' (hourly mean of the two); 12 stations -> 11 sites -> 5,544 rows | Same location under two names; keeping both would count the site twice
Stations now: ['anand_vihar', 'dwarka_sec8', 'ito', 'jahangirpuri', 'lodhi_road_imd', 'najafgarh', 'narela', 'okhla_ph2', 'punjabi_bagh', 'pusa', 'rk_puram']


## 4 · Outliers: detect, then diagnose each as an error or a real value
We use four checks, each with a fixed rule:

| Check | Rule | Likely meaning |
|---|---|---|
| Impossible value | PM2.5 ≤ 0 µg/m³ | Delhi's air is never perfectly clean, and negative mass cannot exist: error |
| Stuck value | Same PM2.5 value repeated for 4 or more hours in a row | Frozen sensor: error |
| Isolated spike | Station value > 3 × its own 24-hour rolling median **and** > 3 × the median of all other stations in that hour | Only one station sees it: sensor fault |
| City-wide high | Value > 3 × own rolling median but other stations also high | Real pollution event (e.g. Diwali night): **kept** |

The key point: a real Diwali peak shows up at **many stations at once**; a sensor fault shows up at **one**.

**A rule we first used and then dropped:** we started by treating anything above 1,000 µg/m³ as "outside the instrument's range". The data proved us wrong. On Diwali night 2025, nine stations went above 1,000 *in the same hours* (the highest was 1,901). A fault does not strike nine separate monitors at once, so these are real readings. Values above 1,000 are now judged by the same city-wide test as everything else.

In [13]:
cpcb = cpcb.sort_values(["station_key", "ts_ist"]).reset_index(drop=True)
cpcb["year"] = cpcb.ts_ist.dt.year
cpcb["pm25_raw"] = cpcb["pm25"]          # keep the pre-cleaning value for the before/after plot
cpcb["pm25_flag"] = "ok"

# (1) Impossible values
bad_range = cpcb.pm25 <= 0
print("Values <= 0:", bad_range.sum(), "| by station:", cpcb[bad_range].groupby("station_key").size().to_dict())

# Values above 1000: how many stations are above 1000 in the same hour?
above = cpcb[cpcb.pm25 > 1000]
same_hour = cpcb[cpcb.pm25 > 1000].groupby("ts_ist")["station_key"].nunique()
print(f"Hours with a value above 1000: {len(above)} station-hours; years: {sorted(above.ts_ist.dt.year.unique())}")
print("Stations above 1000 in the same hour (top 5 hours):"); print(same_hour.sort_values(ascending=False).head(5).to_string())

# (2) Stuck values: runs of >= 4 identical, non-missing values
def run_length(s):
    grp = (s != s.shift()).cumsum()
    return s.groupby(grp).transform("size")
cpcb["run"] = cpcb.groupby(["station_key", "year"])["pm25"].transform(run_length)
stuck = (cpcb["run"] >= 4) & cpcb["pm25"].notna()

# (3)/(4) Spikes: compare with own rolling median and with the other stations
cpcb["roll_med"] = cpcb.groupby(["station_key", "year"])["pm25"].transform(lambda s: s.rolling(25, center=True, min_periods=12).median())
city_med = cpcb[cpcb.pm25 > 0].groupby("ts_ist")["pm25"].median().rename("city_med")   # impossible values excluded
cpcb = cpcb.join(city_med, on="ts_ist")
high_vs_self = cpcb.pm25 > 3 * cpcb.roll_med
spike_isolated = high_vs_self & (cpcb.pm25 > 3 * cpcb.city_med)
spike_citywide = high_vs_self & ~spike_isolated

summary = pd.DataFrame({"check": ["value <= 0", "stuck (>=4 h same value)", "isolated spike (one station only)", "high but city-wide (real event)"],
                        "hours_flagged": [bad_range.sum(), stuck.sum(), spike_isolated.sum(), spike_citywide.sum()]})
summary

Values <= 0: 55 | by station: {'ito': 28, 'lodhi_road_imd': 19, 'pusa': 8}
Hours with a value above 1000: 51 station-hours; years: [np.int32(2025)]
Stations above 1000 in the same hour (top 5 hours):
ts_ist
2025-10-21 03:00:00    9
2025-10-21 02:00:00    8
2025-10-21 04:00:00    6
2025-10-21 05:00:00    6
2025-10-21 01:00:00    5


,check,hours_flagged
0,value <= 0,55
1,stuck (>=4 h same value),64
2,isolated spike (one station only),28
3,high but city-wide (real event),154


In [14]:
# Look at every stuck run and every isolated spike before deciding
stuck_runs = (cpcb[stuck].assign(run_id=lambda d: (d.ts_ist.diff() != pd.Timedelta("1h")).cumsum())
              .groupby(["station_key", "run_id"]).agg(start=("ts_ist", "first"), hours=("ts_ist", "size"), value=("pm25", "first"),
                                                      city_median=("city_med", "median")).reset_index(drop=True))
print("Stuck runs:"); print(stuck_runs.to_string())
print("\nIsolated spikes:")
print(cpcb.loc[spike_isolated, ["station_key", "ts_ist", "pm25", "roll_med", "city_med"]].round(1).to_string())

Stuck runs:
                start  hours   value  city_median
0 2024-09-17 03:00:00      4   41.00      49.8775
1 2024-10-06 00:00:00      4   70.00      61.5925
2 2025-09-17 12:00:00      4    0.00      39.8125
3 2023-10-28 12:00:00     12   87.44     110.9375
4 2024-10-04 11:00:00      5   25.36      30.0000
5 2025-11-25 20:00:00     17    0.00     218.4300
6 2023-09-17 04:00:00      4   20.00      24.8750
7 2023-11-05 19:00:00      4  348.00     335.3750
8 2024-11-01 21:00:00      6  364.00     281.1325
9 2025-10-19 23:00:00      4  379.00     299.5000

Isolated spikes:
          station_key              ts_ist    pm25  roll_med  city_med
652       anand_vihar 2023-10-12 04:00:00   527.2     107.2     147.8
984       anand_vihar 2023-10-26 00:00:00   613.0     145.0     147.2
1008      anand_vihar 2023-10-27 00:00:00   534.5     160.5     164.0
2177      anand_vihar 2024-09-28 17:00:00   220.0      37.0      25.5
2178      anand_vihar 2024-09-28 18:00:00   222.0      38.0      28.5


/var/folders/s3/dpyxt48j22bd6y1q0x3_rft40000gn/T/ipykernel_7911/708904836.py:7: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  print(cpcb.loc[spike_isolated, ["station_key", "ts_ist", "pm25", "roll_med", "city_med"]].round(1).to_string())


**Diagnosis**
- **Values ≤ 0** (negatives down to about −230, and runs of exactly 0.00 at ITO and Lodhi Road in 2025): physically impossible. → **error, set to missing**.
- **Stuck runs:** a PM2.5 monitor reporting the *exact* same value for 4+ hours while the rest of the city changes is a frozen sensor or a value carried forward by the logger, not real air. Some of these sit on Diwali nights (e.g. Punjabi Bagh frozen at 364 for six hours on 1–2 Nov 2024). The level looks plausible, but a flat line hides the real rise and fall of the night, so we cannot trust the timing. → **error, set to missing**; the other ten stations still cover those hours.
- **Isolated spikes:** one station jumps to more than three times its own level and three times the city median, and nobody else sees it. Real pollution plumes in Delhi are wide; a single-station, single-hour jump points to a local fault or a very local source (e.g. a fire next to the inlet). → **error, set to missing**.
- **City-wide highs** (Diwali nights, the 18 Nov 2024 smog episode) → **real, kept unchanged**. These are exactly what the project is about.

In [15]:
# Order matters: 'kept' is written first, so an error flag always wins if a value meets both.
cpcb.loc[spike_citywide, "pm25_flag"] = "citywide_high_kept"
cpcb.loc[spike_isolated, "pm25_flag"] = "isolated_spike"
cpcb.loc[stuck, "pm25_flag"] = "stuck"
cpcb.loc[bad_range, "pm25_flag"] = "impossible_value"
what = {"impossible_value": "Outliers: removed PM2.5 values of zero or below (set to missing)",
        "stuck": "Outliers: removed stuck-sensor runs, the same value 4+ hours in a row (set to missing)",
        "isolated_spike": "Outliers: removed single-station spikes, > 3x the station's 24-h median and > 3x the other stations that hour (set to missing)"}
for flag, why in [("impossible_value", "PM2.5 cannot be zero or negative: an instrument error"), ("stuck", "Real air does not hold one exact value for hours: a frozen sensor"),
                  ("isolated_spike", "Only one station saw it, so it is a local fault, not a pollution event")]:
    n = (cpcb.pm25_flag == flag).sum()
    cpcb.loc[cpcb.pm25_flag == flag, "pm25"] = np.nan
    log("Outliers", what[flag].replace("Outliers: ", ""), n, why)
log("Outliers", "Kept extreme values that many stations recorded at the same time (flagged citywide_high_kept, not changed)", (cpcb.pm25_flag == "citywide_high_kept").sum(),
    "Real events (Diwali nights, smog episodes): exactly what the project studies")

[16] Outliers: removed PM2.5 values of zero or below (set to missing) -> 55 rows | PM2.5 cannot be zero or negative: an instrument error
[17] Outliers: removed stuck-sensor runs, the same value 4+ hours in a row (set to missing) -> 43 rows | Real air does not hold one exact value for hours: a frozen sensor
[18] Outliers: removed single-station spikes, > 3x the station's 24-h median and > 3x the other stations that hour (set to missing) -> 28 rows | Only one station saw it, so it is a local fault, not a pollution event
[19] Outliers: Kept extreme values that many stations recorded at the same time (flagged citywide_high_kept, not changed) -> 154 rows | Real events (Diwali nights, smog episodes): exactly what the project studies


In [16]:
# A physical consistency check: PM2.5 is a subset of PM10, so PM2.5 should not exceed PM10 by much.
over = (cpcb.pm25 > cpcb.pm10 + 10) & cpcb.pm10.notna()
print("Hours with PM2.5 > PM10 + 10:", over.sum(), "of", cpcb.pm10.notna().sum())
print(cpcb.loc[over].groupby("station_key").size().sort_values(ascending=False).head(6).to_string())
log("Outliers", "Checked hours where PM2.5 is more than 10 µg/m³ above PM10; left unchanged", over.sum(),
    "Impossible if both are right, but they come from separate instruments and we cannot tell which is wrong; PM10 is not used in the analysis")

Hours with PM2.5 > PM10 + 10: 70 of 58612
station_key
jahangirpuri      33
pusa              26
ito                4
rk_puram           3
anand_vihar        1
lodhi_road_imd     1
[20] Outliers: Checked hours where PM2.5 is more than 10 µg/m³ above PM10; left unchanged -> 70 rows | Impossible if both are right, but they come from separate instruments and we cannot tell which is wrong; PM10 is not used in the analysis


## 5 · Missing values
### 5a · How much is missing, and where

In [17]:
miss = cpcb.groupby(["station_key", "year"])["pm25"].apply(lambda s: s.isna().mean() * 100).unstack().round(1)
miss["all_years"] = cpcb.groupby("station_key")["pm25"].apply(lambda s: s.isna().mean() * 100).round(1)
print("Percent of hours with no PM2.5 (after outlier removal):")
miss.sort_values("all_years", ascending=False)

Percent of hours with no PM2.5 (after outlier removal):


year,2023,2024,2025,all_years
station_key,,,,
anand_vihar,10.9,11.7,2.9,8.5
lodhi_road_imd,1.0,1.2,16.0,6.1
rk_puram,4.2,7.5,1.8,4.5
punjabi_bagh,1.0,2.4,8.3,3.9
najafgarh,0.4,4.0,6.2,3.5
ito,1.2,7.1,1.6,3.3
okhla_ph2,1.1,1.0,4.2,2.1
jahangirpuri,1.6,1.1,1.8,1.5
dwarka_sec8,0.5,1.0,2.0,1.2


In [18]:
# Pattern 1: gap lengths. Most gaps are 1-2 hours; a few last days.
def gap_lengths(s):
    isna = s.isna()
    grp = (isna != isna.shift()).cumsum()
    return isna.groupby(grp).sum()[lambda x: x > 0]
gaps = pd.concat([gap_lengths(g.set_index("ts_ist")["pm25"]) for _, g in cpcb.groupby(["station_key", "year"])])
gap_table = pd.cut(gaps, [0, 1, 3, 6, 24, 10_000], labels=["1 h", "2-3 h", "4-6 h", "7-24 h", "> 24 h"]).value_counts().sort_index()
gap_table.to_frame("number of gaps").assign(hours_lost=gaps.groupby(pd.cut(gaps, [0, 1, 3, 6, 24, 10_000])).sum().values)

,number of gaps,hours_lost
pm25,,
1 h,233,233
2-3 h,111,250
4-6 h,40,184
7-24 h,57,728
> 24 h,8,574


In [19]:
# Pattern 2: by hour of day (if missingness were completely random, every hour would have about the same share)
by_hour = cpcb.groupby(cpcb.ts_ist.dt.hour)["pm25"].apply(lambda s: s.isna().mean() * 100).round(2)
by_hour.to_frame("% missing").T

ts_ist,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
% missing,3.31,3.42,3.62,3.78,3.54,3.78,3.58,3.31,3.19,3.19,3.15,3.03,3.5,3.46,3.82,3.46,3.07,2.79,3.15,2.75,2.6,2.72,2.79,2.48


### 5b · Why is it missing? (MCAR, MAR or MNAR)
- **MCAR** (missing completely at random): gaps would be unrelated to anything, including time of day and pollution level.
- **MAR** (missing at random): gaps depend on something we *do* observe, e.g. a particular station, hour or the city-wide pollution level.
- **MNAR** (missing not at random): gaps depend on the missing value itself, e.g. the monitor stops when the air is extremely dirty.

**Test 1 (chi-square):** is missingness independent of the hour of day?
**Test 2 (Mann–Whitney U):** when a station is missing, is the *rest of the city* more or less polluted than when it reports? We cannot see the missing station's own value, but the city median that hour is a good proxy for it. If gaps happened mostly in very dirty hours, that would point to MNAR.

In [20]:
ct = pd.crosstab(cpcb.ts_ist.dt.hour, cpcb.pm25.isna())
chi2, p_hour, dof, _ = stats.chi2_contingency(ct)
print(f"Test 1 - missing vs hour of day: chi2 = {chi2:.1f}, dof = {dof}, p = {p_hour:.3g}")

ct_st = pd.crosstab(cpcb.station_key, cpcb.pm25.isna())
chi2s, p_st, dofs, _ = stats.chi2_contingency(ct_st)
print(f"        missing vs station:     chi2 = {chi2s:.1f}, dof = {dofs}, p = {p_st:.3g}")

city_when_missing = cpcb.loc[cpcb.pm25.isna(), "city_med"].dropna()
city_when_present = cpcb.loc[cpcb.pm25.notna(), "city_med"].dropna()
u, p_mw = stats.mannwhitneyu(city_when_missing, city_when_present)
log("Missing values", f"Tested why PM2.5 is missing: chi-square by hour of day p = {p_hour:.2f}, by station p < 0.001; city median when missing {city_when_missing.median():.0f} vs {city_when_present.median():.0f} µg/m³ (Mann-Whitney p < 0.001). Judged MAR",
    cpcb.pm25.isna().sum(), "Gaps depend on station and season (both observed), not on the missing value itself")
print(f"Test 2 - city median PM2.5 when station missing: {city_when_missing.median():.0f} vs reporting: {city_when_present.median():.0f} "
      f"(Mann-Whitney p = {p_mw:.3g})")

Test 1 - missing vs hour of day: chi2 = 28.2, dof = 23, p = 0.208
        missing vs station:     chi2 = 1099.4, dof = 10, p = 6.96e-230
[21] Missing values: Tested why PM2.5 is missing: chi-square by hour of day p = 0.21, by station p < 0.001; city median when missing 75 vs 119 µg/m³ (Mann-Whitney p < 0.001). Judged MAR -> 1,969 rows | Gaps depend on station and season (both observed), not on the missing value itself
Test 2 - city median PM2.5 when station missing: 75 vs reporting: 119 (Mann-Whitney p = 3.21e-55)


In [21]:
by_month = cpcb.groupby(cpcb.ts_ist.dt.month)["pm25"].apply(lambda s: s.isna().mean() * 100).round(1)
print("% missing by month (9 = Sep, 10 = Oct, 11 = Nov):", by_month.to_dict())

% missing by month (9 = Sep, 10 = Oct, 11 = Nov): {9: 5.3, 10: 3.6, 11: 1.8}


**Our reading of the tests**
- **Hour of day does not matter** (chi-square p ≈ 0.2): monitors are not switched off at night or at rush hour.
- **Station matters a lot** (p ≈ 0): Anand Vihar and Lodhi Road lose far more hours than Pusa or Narela.
- **Season matters:** gaps are most common in September (≈5.3% of hours) and rarest in November (≈1.8%). That is why the city is *cleaner* during gaps (median 75 vs 119 µg/m³, Mann–Whitney p ≈ 0): the gaps fall in the cleaner part of the season, not because monitors fail in clean air.
- So the gaps are **not completely random (not MCAR)**. But everything they depend on (station and date) is something we observe, and there is no sign that monitors drop out *because* the air is extremely dirty: if anything, dirty November hours are the best covered. We therefore treat the data as **MAR**.
- What this means for us: averages that pool many hours could be pulled slightly upward, because clean September hours are missing more often. We handle that by comparing like with like (same station, same part of the season) and by requiring 75% coverage for any daily average.

### 5c · What we do about it
- **Short gaps (≤ 3 hours): linear interpolation in time.** PM2.5 changes smoothly from hour to hour, so a straight line between the hour before and the hour after is a good estimate, and under MAR it does not bias the result. Each filled value is marked `pm25_source = "interpolated"`.
- **Longer gaps: left missing.** Filling a whole day would invent data, and it could erase or create a Diwali peak. Deleting these hours is safe under MAR because we average over many stations and many hours.
- **Daily or nightly averages** are only computed when at least 75% of hours are present.

In [22]:
def interp_short(s, limit=3):
    """Interpolate gaps of at most `limit` hours; leave longer gaps empty."""
    isna = s.isna()
    grp = (isna != isna.shift()).cumsum()
    gap_len = isna.groupby(grp).transform("sum")
    filled = s.interpolate(method="linear", limit_area="inside")
    return s.where(~(isna & (gap_len <= limit)), filled)

cpcb.loc[cpcb.pm25.isna(), "pm25_source"] = None   # removed outliers no longer have a source
before = cpcb.pm25.isna().sum()
cpcb["pm25"] = cpcb.groupby(["station_key", "year"])["pm25"].transform(interp_short)
new_fill = cpcb.pm25.notna() & cpcb.pm25_source.isna()
cpcb.loc[new_fill, "pm25_source"] = "interpolated"
log("Missing values", "Filled PM2.5 gaps of 3 hours or less by straight-line interpolation", new_fill.sum(), "PM2.5 changes smoothly hour to hour, and the gaps are MAR")
log("Missing values", "Left PM2.5 gaps longer than 3 hours empty (2.4% of hours)", cpcb.pm25.isna().sum(), "Filling long gaps would invent data and could hide or create a Diwali peak; safe to leave out because the gaps are MAR")
print(f"Missing PM2.5 before: {before:,}  after: {cpcb.pm25.isna().sum():,}")

[22] Missing values: Filled PM2.5 gaps of 3 hours or less by straight-line interpolation -> 482 rows | PM2.5 changes smoothly hour to hour, and the gaps are MAR
[23] Missing values: Left PM2.5 gaps longer than 3 hours empty (2.4% of hours) -> 1,487 rows | Filling long gaps would invent data and could hide or create a Diwali peak; safe to leave out because the gaps are MAR
Missing PM2.5 before: 1,969  after: 1,487


## 6 · Station weather sensors vs Open-Meteo (a quality check, not used for the analysis)
The CPCB station sensors give a ground-truth check on the modelled weather. We also look for obviously stuck weather sensors.

In [23]:
st_city = cpcb.groupby("ts_ist")[["st_temp_c", "st_wind_kmh", "st_rh_pct"]].median()
cmp = wx.set_index("ts_ist").join(st_city, how="inner")
check = pd.DataFrame({
    "variable": ["temperature (°C)", "wind speed (km/h)", "humidity (%)"],
    "Open-Meteo mean": [cmp.temp_c.mean(), cmp.wind_kmh.mean(), cmp.rh_pct.mean()],
    "CPCB stations (median) mean": [cmp.st_temp_c.mean(), cmp.st_wind_kmh.mean(), cmp.st_rh_pct.mean()],
    "correlation": [cmp[["temp_c", "st_temp_c"]].corr().iloc[0, 1], cmp[["wind_kmh", "st_wind_kmh"]].corr().iloc[0, 1],
                    cmp[["rh_pct", "st_rh_pct"]].corr().iloc[0, 1]]}).round(2)
check

,variable,Open-Meteo mean,CPCB stations (median) mean,correlation
0,temperature (°C),24.60,26.64,0.87
1,wind speed (km/h),7.05,1.65,0.60
2,humidity (%),62.99,65.55,0.84


In [24]:
wind_floor = cpcb.st_wind_kmh.round(2).value_counts().head(3)
print("Most common CPCB wind values (km/h):", wind_floor.to_dict())
print("Share of station wind readings at the single most common value:", round(wind_floor.iloc[0] / cpcb.st_wind_kmh.notna().sum() * 100, 1), "%")
rh_bad = (cpcb.st_rh_pct > 100).sum()
print("Station humidity above 100%:", rh_bad)
t_bad, w_bad = (cpcb.st_temp_c > 45).sum(), (cpcb.st_wind_kmh > 60).sum()
print("Station temperature above 45 °C:", t_bad, "| station wind above 60 km/h:", w_bad)
log("Outliers", f"Checked station weather sensors: {t_bad} temperatures above 45 °C and {w_bad} winds above 60 km/h; left unchanged", t_bad + w_bad,
    "Station sensors are only used to check Open-Meteo, never in the analysis")

Most common CPCB wind values (km/h): {1.08: 19515, 1.26: 1412, 1.15: 1409}
Share of station wind readings at the single most common value: 39.9 %
Station humidity above 100%: 0
Station temperature above 45 °C: 11 | station wind above 60 km/h: 57
[24] Outliers: Checked station weather sensors: 11 temperatures above 45 °C and 57 winds above 60 km/h; left unchanged -> 68 rows | Station sensors are only used to check Open-Meteo, never in the analysis


Temperature agrees closely. Station wind speeds are much lower than the model's: many CPCB anemometers sit low among buildings and trees, and a large share of readings sit on a single floor value (the sensor's minimum). That is why we use **Open-Meteo** for the weather analysis and keep the station weather only as a check.

## 7 · Build the final table, save it, read it back

In [25]:
final = (cpcb.merge(wx, on="ts_ist", how="left")
              .assign(year=lambda d: d.ts_ist.dt.year,
                      diwali_date=lambda d: pd.to_datetime(d.year.map(DIWALI)),
                      days_from_diwali=lambda d: (d.ts_ist.dt.normalize() - d.diwali_date).dt.days)
              .drop(columns=["diwali_date", "run", "roll_med", "city_med", "cpcb_site_id"]))
final["pm25_source"] = final["pm25_source"].fillna("missing")
cols = ["station_key", "ts_ist", "year", "days_from_diwali", "pm25", "pm25_raw", "pm25_flag", "pm25_source", "pm10",
        "st_temp_c", "st_rh_pct", "st_wind_kmh", "st_wind_dir_deg",
        "temp_c", "rh_pct", "rain_mm", "wind_kmh", "wind_dir_deg", "pressure_hpa", "blh_m"]
final = final[cols].sort_values(["station_key", "ts_ist"]).reset_index(drop=True)
final["station_key"] = final["station_key"].astype("category")
final["pm25_flag"] = final["pm25_flag"].astype("category")
final["pm25_source"] = final["pm25_source"].astype("category")
final["year"] = final["year"].astype("int16"); final["days_from_diwali"] = final["days_from_diwali"].astype("int16")

OUT = CLEAN / "delhi_pm25_hourly.parquet"
final.to_parquet(OUT, index=False)
back = pd.read_parquet(OUT)
assert back.dtypes.equals(final.dtypes) and back.shape == final.shape
print("Saved and re-read:", OUT.name, back.shape)
print(back.dtypes.to_string())

Saved and re-read: delhi_pm25_hourly.parquet (60984, 20)
station_key               category
ts_ist              datetime64[us]
year                         int16
days_from_diwali             int16
pm25                       float64
pm25_raw                   float64
pm25_flag                 category
pm25_source               category
pm10                       float64
st_temp_c                  float64
st_rh_pct                  float64
st_wind_kmh                float64
st_wind_dir_deg            float64
temp_c                     float64
rh_pct                       int64
rain_mm                    float64
wind_kmh                   float64
wind_dir_deg                 int64
pressure_hpa               float64
blh_m                      float64


**Why Parquet:** it stores each column's type with the data (datetimes stay datetimes, categories stay categories), so nothing has to be re-parsed after reading. That is exactly what the read-back check above confirms. It is also about 10× smaller than the same data as CSV. A CSV copy is saved too, for anyone who wants to open the table in Excel.

In [26]:
final.to_csv(CLEAN / "delhi_pm25_hourly.csv", index=False)
print("Rows per station:", final.groupby("station_key", observed=True).size().unique(),
      "| stations:", final.station_key.nunique(), "| years:", sorted(final.year.unique()))
final.pm25_source.value_counts()

Rows per station: [5544] | stations: 11 | years: [np.int16(2023), np.int16(2024), np.int16(2025)]


pm25_source
cpcb            58892
missing          1487
interpolated      482
openaq            123
Name: count, dtype: int64

## 8 · Data dictionary and cleaning log

In [27]:
# column: (meaning, unit, source, allowed range, note)
DICT = {
    "station_key": ("Monitoring site (11 sites; the two Pusa monitors merged into 'pusa')", "", "CPCB station list", "one of 11 site names", ""),
    "ts_ist": ("Hour the reading covers (start of the hour)", "IST", "CPCB timestamp", "15 Sep - 30 Nov, 2023-2025", "1,848 hours per site per year"),
    "year": ("Season year", "", "derived from ts_ist", "2023-2025", ""),
    "days_from_diwali": ("Days before (-) or after (+) that year's Diwali", "days", "derived from ts_ist", "-58 to +41", "Diwali: 12 Nov 2023, 31 Oct 2024, 20 Oct 2025"),
    "pm25": ("Hourly PM2.5 after cleaning; used in all analysis", "µg/m³", "CPCB repository (gaps in 2025 filled from OpenAQ)", "> 0, no upper limit", "Missing hours are gaps longer than 3 hours, left empty on purpose"),
    "pm25_raw": ("Hourly PM2.5 before outlier removal and gap filling", "µg/m³", "CPCB repository / OpenAQ", "any (still contains the errors)", "Kept for the before/after comparison"),
    "pm25_flag": ("Result of the outlier checks for that hour", "", "derived", "ok, impossible_value, stuck, isolated_spike, citywide_high_kept", "First three were set to missing in pm25"),
    "pm25_source": ("Where the cleaned pm25 value came from", "", "derived", "cpcb, openaq, interpolated, missing", ""),
    "pm10": ("Hourly PM10; context only, not used in the analysis", "µg/m³", "CPCB repository", "> 0", "Never above exactly 1,000: readings look capped by the instrument"),
    "st_temp_c": ("Air temperature measured at the station", "°C", "CPCB repository (station sensor)", "0 - 45", "Check only; some readings above 45 left as recorded"),
    "st_rh_pct": ("Relative humidity measured at the station", "%", "CPCB repository (station sensor)", "0 - 100", "Check only"),
    "st_wind_kmh": ("Wind speed measured at the station", "km/h", "CPCB repository (station sensor, converted from m/s)", "0 - 60", "Check only; 40% of readings sit on the sensor minimum, some above 60"),
    "st_wind_dir_deg": ("Wind direction measured at the station", "degrees", "CPCB repository (station sensor)", "0 - 360", "Check only"),
    "temp_c": ("Air temperature 2 m above ground", "°C", "Open-Meteo Historical Weather API", "0 - 45", "Same for every site in a given hour"),
    "rh_pct": ("Relative humidity 2 m above ground", "%", "Open-Meteo Historical Weather API", "0 - 100", ""),
    "rain_mm": ("Rain in that hour", "mm", "Open-Meteo Historical Weather API", "0 - 50", ""),
    "wind_kmh": ("Wind speed 10 m above ground", "km/h", "Open-Meteo Historical Weather API", "0 - 60", ""),
    "wind_dir_deg": ("Direction the wind blows from (0 = north)", "degrees", "Open-Meteo Historical Weather API", "0 - 360", ""),
    "pressure_hpa": ("Air pressure at the surface", "hPa", "Open-Meteo Historical Weather API", "950 - 1010", ""),
    "blh_m": ("Boundary-layer height: depth of the air layer pollution mixes into", "m", "Open-Meteo Historical Weather API", "0 - 3000", ""),
}
def rng(c):
    if back[c].dtype.kind not in "iuf":
        return ""
    lo, hi = back[c].min(), back[c].max()
    f = lambda v: f"{v:.0f}" if float(v).is_integer() or abs(v) >= 100 else f"{v:.1f}"
    return f"{f(lo)} to {f(hi)}"
dd = pd.DataFrame([{"column": c, "meaning": DICT[c][0], "dtype": str(back[c].dtype), "unit": DICT[c][1], "source": DICT[c][2],
                    "allowed_range": DICT[c][3], "observed_range": rng(c), "missing_pct": round(back[c].isna().mean() * 100, 1), "note": DICT[c][4]}
                   for c in back.columns])
dd.to_csv(META / "data_dictionary.csv", index=False)
dd

,column,meaning,dtype,unit,source,allowed_range,observed_range,missing_pct,note
0,station_key,Monitoring site (11 sites; the two Pusa monito...,category,,CPCB station list,one of 11 site names,,0.0,
1,ts_ist,Hour the reading covers (start of the hour),datetime64[us],IST,CPCB timestamp,"15 Sep - 30 Nov, 2023-2025",,0.0,"1,848 hours per site per year"
2,year,Season year,int16,,derived from ts_ist,2023-2025,2023 to 2025,0.0,
3,days_from_diwali,Days before (-) or after (+) that year's Diwali,int16,days,derived from ts_ist,-58 to +41,-58 to 41,0.0,"Diwali: 12 Nov 2023, 31 Oct 2024, 20 Oct 2025"
4,pm25,Hourly PM2.5 after cleaning; used in all analysis,float64,µg/m³,CPCB repository (gaps in 2025 filled from OpenAQ),"> 0, no upper limit",0.4 to 1901,2.4,"Missing hours are gaps longer than 3 hours, le..."
5,pm25_raw,Hourly PM2.5 before outlier removal and gap fi...,float64,µg/m³,CPCB repository / OpenAQ,any (still contains the errors),-233 to 1901,3.0,Kept for the before/after comparison
6,pm25_flag,Result of the outlier checks for that hour,category,,derived,"ok, impossible_value, stuck, isolated_spike, c...",,0.0,First three were set to missing in pm25
7,pm25_source,Where the cleaned pm25 value came from,category,,derived,"cpcb, openaq, interpolated, missing",,0.0,
8,pm10,"Hourly PM10; context only, not used in the ana...",float64,µg/m³,CPCB repository,> 0,0.6 to 1000,3.9,"Never above exactly 1,000: readings look cappe..."
9,st_temp_c,Air temperature measured at the station,float64,°C,CPCB repository (station sensor),0 - 45,2.2 to 50.8,19.7,Check only; some readings above 45 left as rec...


In [28]:
cleaning_log = pd.DataFrame(LOG)
cleaning_log.to_csv(META / "cleaning_log.csv", index=False)
cleaning_log

,step,stage,what_we_did,rows_affected,why
0,1,Formats,CPCB: renamed the 7 columns we keep from 'NAME...,315648,"Units go in the data dictionary, not in column..."
1,2,Formats,CPCB: converted the Timestamp text to dates (I...,315648,Text dates cannot be sorted or joined
2,3,Scope,CPCB: removed rows outside 15 Sep - 30 Nov (31...,249120,Outside the study window
3,4,Scope,"OpenAQ: removed readings of other parameters, ...",723420,Other pollutants and weather are outside the q...
4,5,Formats,OpenAQ: converted timestamps with a +05:30 off...,80525,The two sources must use the same clock before...
5,6,Formats,"OpenAQ: cleaned station names, removing the nu...",80525,So names can be compared with CPCB's spelling
6,7,Duplicates,OpenAQ: checked for repeated readings (same se...,0,A reading should appear once
7,8,Scope,OpenAQ: removed readings outside 15 Sep - 30 N...,1981,Outside the study window
8,9,Formats,OpenAQ: averaged 15-minute readings into hourl...,56639,CPCB reports hourly averages; both sources nee...
9,10,Units,CPCB: converted station wind speed from m/s to...,48865,Open-Meteo wind is in km/h; one unit per quantity
